In [4]:
import sys
from pathlib import Path

#仓库根：从当前目录往上找带 config.py 的目录（git clone 和直接下载 zip 都能用）
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

import os, re, glob
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.inchi import MolToInchi
from itertools import combinations

## 去除重复分子
1. 重复分子只保留下标最小的那条，其余删除
2. 输出 xlsx 里只有去重后的分子（不含重复分子，也不含被删下标）
3. 被去除分子的原下标用 print 输出到控制台

In [5]:
RDLogger.DisableLog("rdApp.*")            # 关掉 RDKit 警告刷屏

# ============== 路径（统一走仓库根的 config.py，不再写死绝对路径） ==============
IN_DIR   = config.DATA_TDT    #仓库里的 TDT/data
IN_FILE  = "test.xlsx"
OUT_DIR  = config.OUT         #产物默认写到 TDT/out，想改就设环境变量 CHEM_OUT
OUT_FILE = "去重结果_test.xlsx"
# =======================================================================

LEVEL = "safe"

COL_INDEX, COL_TD, COL_SMILES = None, None, None

SMILES_CHARS = re.compile(r"^[0-9A-Za-z@+\-\[\]\(\)=#$:/\\.%*]+$")
HEADER_WORDS = {"smiles", "canonical_smiles", "结构", "分子", "td(°c)"}


def find_input():
    p = os.path.join(IN_DIR, IN_FILE)
    if os.path.isfile(p):
        return p
    cands = sorted(glob.glob(os.path.join(IN_DIR, "*.xlsx")))
    if not cands:
        raise FileNotFoundError(f"在 {IN_DIR} 里找不到 xlsx 文件")
    print(f"提示：没找到 {IN_FILE}，改用 {os.path.basename(cands[0])}")
    return cands[0]


def read_table(path):
    """读 xlsx, 自动识别「下标 / Td / SMILES」三列；自动跳过表头行"""
    raw = pd.read_excel(path, header=None, dtype=str).fillna("").astype(str)

    # 改①：先剔表头行 —— 表头既不参与列识别，也不该计入分子数
    hdr = [i for i in range(len(raw))
           if any(v.strip().lower() in HEADER_WORDS for v in raw.iloc[i])]
    data = raw.drop(index=hdr).reset_index(drop=True)
    n = len(data)                                   # 真实数据行数

    # 改②：列号为 None 时返回空串，不再让 iloc 崩
    def col(j):
        if j is None:
            return [""] * n
        return [v.strip() for v in data.iloc[:, j]]

    # 1) SMILES 列：符合 SMILES 字符集且含 = [ ] ( ) 的比例最高
    best_smi, best_score = None, -1.0
    for j in range(data.shape[1]):
        vals = [v for v in col(j) if v]
        if not vals:
            continue
        ok = sum(1 for v in vals if SMILES_CHARS.match(v) and re.search(r"[=\[\]()]", v))
        score = ok / len(vals)
        if score > best_score:
            best_smi, best_score = j, score

    # 2) 下标列：全为整数、唯一、排序后正好是 0..n-1
    best_idx = None
    for j in range(data.shape[1]):
        if j == best_smi:
            continue
        vals = [v for v in col(j) if v != ""]
        if len(vals) == n and all(re.fullmatch(r"\d+", v) for v in vals) \
                and sorted(int(v) for v in vals) == list(range(n)):
            best_idx = j
            break
    if best_idx is None:
        for j in range(data.shape[1]):
            if j == best_smi:
                continue
            vals = [v for v in col(j) if v != ""]
            if vals and all(re.fullmatch(r"-?\d+", v) for v in vals):
                best_idx = j
                break

    # 3) Td 列：剩下列里数值比例最高的
    best_td, best_td_score = None, -1.0
    for j in range(data.shape[1]):
        if j in (best_smi, best_idx):
            continue
        vals = [v for v in col(j) if v != ""]
        if not vals:
            continue
        score = sum(1 for v in vals if re.fullmatch(r"-?\d+(\.\d+)?", v)) / len(vals)
        if score > best_td_score:
            best_td, best_td_score = j, score

    if COL_SMILES is not None: best_smi = COL_SMILES
    if COL_INDEX  is not None: best_idx = COL_INDEX
    if COL_TD     is not None: best_td  = COL_TD

    print(f"[读表] 总行数={len(raw)}  表头行={hdr}  数据行={n}  列数={data.shape[1]}  "
          f"识别列: idx={best_idx}, td={best_td}, smi={best_smi}")

    ic, tc, sc = col(best_idx), col(best_td), col(best_smi)
    df = pd.DataFrame({
        "下标":   [int(ic[i]) if best_idx is not None and re.fullmatch(r"-?\d+", ic[i]) else i
                   for i in range(n)],
        "Td(°C)": [tc[i] if best_td is not None else "" for i in range(n)],
        "SMILES": sc,
    })
    drop = (df["SMILES"] == "") | df["SMILES"].str.lower().isin(HEADER_WORDS)
    if drop.any():
        print(f"[读表] 再剔除空 SMILES 行（行号 {df.index[drop].tolist()}）")
    return df[~drop].reset_index(drop=True)


def main():
    src = find_input()
    print(f"读取文件：{src}")
    df = read_table(src)
    print(f"读入 {len(df)} 条记录")

    # ---------- 标准化 ----------
    recs, bad = [], []
    for _, row in df.iterrows():
        smi = row["SMILES"]
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            bad.append(row["下标"])
            recs.append({"idx": row["下标"], "td": row["Td(°C)"], "smi": smi,
                         "inchi": f"BAD:{smi}", "cons": f"BAD:{smi}", "annot": False, "can": ""})
            continue
        m2 = Chem.Mol(mol)
        Chem.RemoveStereochemistry(m2)
        # 改③（核心）：立体标注看分子本身，不看原始字符串 —— 这是 69/70 差异的根源
        annot = (any(a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED for a in mol.GetAtoms())
                 or any(b.GetStereo() != Chem.BondStereo.STEREONONE for b in mol.GetBonds()))
        recs.append({
            "idx": row["下标"], "td": row["Td(°C)"], "smi": smi,
            "inchi": MolToInchi(mol) or ("SMI:" + Chem.MolToSmiles(mol)),
            "cons":  Chem.MolToSmiles(m2),
            "annot": annot,
            "can":   Chem.MolToSmiles(mol),
        })
    if bad:
        print("警告：以下下标无法解析 SMILES, 已原样保留、未参与去重：", bad)

    # ---------- 去重 ----------
    removed, review = {}, []

    # 改④：代表只按分子内容选，不依赖行序 → 打乱顺序结果一致
    def rep_of(grp):
        return min(grp, key=lambda r: (r["can"], r["idx"]))

    # ① 标准 InChI 完全一致
    g = {}
    for r in recs:
        g.setdefault(r["inchi"], []).append(r)
    for grp in g.values():
        if len(grp) > 1:
            k = rep_of(grp)
            for r in grp:
                if r is not k:
                    removed[r["idx"]] = (k["idx"], "InChI一致")

    # ② / ③ 同一骨架
    if LEVEL in ("safe", "full"):
        g2 = {}
        for r in recs:
            if r["idx"] not in removed:
                g2.setdefault(r["cons"], []).append(r)
        for grp in g2.values():
            if len(grp) < 2:
                continue
            if LEVEL == "safe":
                ann_inchi = {r["inchi"] for r in grp if r["annot"]}
                if len(ann_inchi) > 1:       # 双方都标了立体且冲突 → 真立体异构体，不删
                    review.append(grp)
                    continue
            k = rep_of(grp)
            for r in grp:
                if r is not k:
                    removed[r["idx"]] = (k["idx"], "骨架相同/立体标注缺失")

    by_idx = {r["idx"]: r for r in recs}
    kept = [r for r in recs if r["idx"] not in removed]

    # ---------- 自检 ----------
    chk = {}
    for r in kept:
        chk.setdefault(r["inchi"], []).append(r["idx"])
    left = {k: v for k, v in chk.items() if len(v) > 1}
    assert not left, f"自检失败：保留集里仍有重复 {left}"

    # ---------- 输出 ----------
    def num(v):
        try:
            return float(v)
        except (TypeError, ValueError):
            return v

    out_rows = [[r["idx"], num(r["td"]), r["smi"], r["can"]]
                for r in sorted(kept, key=lambda x: x["idx"])]
    out_df = pd.DataFrame(out_rows, columns=["下标", "Td(°C)", "SMILES", "标准化SMILES"])

    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = os.path.join(OUT_DIR, OUT_FILE)
    with pd.ExcelWriter(out_path, engine="openpyxl") as w:
        out_df.to_excel(w, sheet_name="去重后", index=False)

    # ---------- 控制台输出 ----------
    rm_sorted = sorted(removed)
    print("\n" + "=" * 70)
    print(f"原始分子数：{len(recs)}    保留：{len(kept)}    去除：{len(rm_sorted)}")
    print(f"去重强度 LEVEL = {LEVEL}")
    print("-" * 70)
    print(f"被去除的分子原下标（共 {len(rm_sorted)} 个）：")
    print(", ".join(str(i) for i in rm_sorted))
    print("-" * 70)
    print("对应关系（保留下标 -> 被删下标）：")
    bucket = {}
    for i in rm_sorted:
        bucket.setdefault(removed[i][0], []).append(i)
    for keep_i in sorted(bucket):
        print(f"  {keep_i}  <-  {', '.join(str(x) for x in bucket[keep_i])}")

    if review:
        print("-" * 70)
        print("以下骨架相同、但立体构型互相冲突（判为不同化合物，未删除，请人工确认）：")
        for grp in review:
            print("  " + " | ".join(f"{r['idx']}(Td={r['td']})" for r in grp))

    print("=" * 70)
    print(f"结果已保存：{out_path}")


if __name__ == "__main__":
    main()

读取文件：C:\Users\czywg\Desktop\Programmer_czy\chem_project\TDT\data\test.xlsx
[读表] 总行数=1505  表头行=[0]  数据行=1504  列数=2  识别列: idx=None, td=0, smi=1
读入 1504 条记录

原始分子数：1504    保留：1434    去除：70
去重强度 LEVEL = safe
----------------------------------------------------------------------
被去除的分子原下标（共 70 个）：
18, 19, 70, 183, 237, 282, 311, 333, 345, 374, 378, 387, 433, 451, 500, 518, 531, 545, 566, 576, 590, 595, 600, 601, 637, 643, 666, 721, 724, 731, 762, 774, 792, 796, 810, 850, 858, 862, 882, 887, 905, 926, 987, 1019, 1056, 1061, 1068, 1080, 1113, 1177, 1179, 1187, 1209, 1220, 1232, 1267, 1283, 1285, 1332, 1364, 1371, 1399, 1401, 1403, 1436, 1440, 1442, 1451, 1471, 1483
----------------------------------------------------------------------
对应关系（保留下标 -> 被删下标）：
  11  <-  18
  72  <-  70
  90  <-  19
  184  <-  183
  312  <-  311
  334  <-  333
  341  <-  345
  375  <-  374
  382  <-  378
  388  <-  387
  434  <-  433
  436  <-  237
  497  <-  500
  514  <-  518
  526  <-  595
  529  <-  531
  542  <

## SMILES 修复

由于一些历史原因，上一个 cell 去除了 70 条重复，但是本 cell 用的是只去除了 69 条的数据，这是因为之前的代码问题。

但是在使用这篇 cell 进行 SMILES 修复并进一步去重之后，出现偏差的那一条被顺利剔除，不对最终结果产生影响。

In [6]:
# ============ 0. 路径（按需修改） ============
RDLogger.DisableLog("rdApp.*")                 # 关掉 InChI/立体 警告刷屏

SRC = config.DATA_TDT / "unique_random.xlsx"   # 输入
OUT = config.OUT / "去重修复结果.xlsx"     # 输出
assert SRC.exists(), f"找不到输入文件：{SRC}"


# ============ 1. 读 + 诊断 ============
df = pd.read_excel(SRC)

def diag(s):
    """返回 (净电荷, 未成对电子)，解析失败给 None"""
    m = Chem.MolFromSmiles(s)
    if m is None:
        return pd.Series({"q": None, "rad": None})
    return pd.Series({"q": Chem.GetFormalCharge(m),
                      "rad": sum(a.GetNumRadicalElectrons() for a in m.GetAtoms())})

df[["q", "rad"]] = df["SMILES"].apply(diag)
df["ok"] = (df["q"] == 0) & (df["rad"] == 0)      # 中性 + 闭壳层 = 正常
bad = df.index[~df["ok"]]                          # 坏行下标
print(f"共 {len(df)} 条，有问题 {len(bad)} 条")


# ============ 2. 修复（规则子集搜索） ============
RULES = [
    ("O=[N+](O)",  "O=[N+]([O-])"),     # 硝基画成 -OH → O-
    ("[N+](=O)O",  "[N+](=O)[O-]"),     # 同上另一写法
    ("O=[N+]=N",   "ON=N"),             # 重氮缺 H
    ("N=[N+]=O",   "N=NO"),
    ("no[n+]1C",   "no[n+]([O-])1"),    # 环上 N-甲基 → N-氧化物
    ("[NH-]",      "N"),                # 补一个 H
    ("[n-]",       "[nH]"),
    ("[NH]",       "N"),
    ("[CH-]",      "C"),
    ("[N][n+]",    "[N-][n+]"),         # 自由基氮 → 负电氮
]

def try_fix(s):
    """枚举所有能用的规则子集，挑「中性闭壳层 + H变化最小 + 规则最少」的解"""
    h0 = Chem.AddHs(Chem.MolFromSmiles(s)).GetNumAtoms()
    cand = [(a, b) for a, b in RULES if a in s]      # 只保留字符串里真出现的规则
    best = None
    for r in range(1, len(cand) + 1):
        for combo in combinations(cand, r):
            t = s
            for a, b in combo:
                t = t.replace(a, b)
            m = Chem.MolFromSmiles(t)
            if m is None:
                continue
            if Chem.GetFormalCharge(m) or sum(a.GetNumRadicalElectrons() for a in m.GetAtoms()):
                continue                                 # 必须中性闭壳层
            dh = abs(Chem.AddHs(m).GetNumAtoms() - h0)
            key = (dh, r)
            if best is None or key < best[0]:
                best = (key, t, combo)
    return (best[1], [a for a, _ in best[2]]) if best else (None, [])

df["SMILES_fix"] = df["SMILES"]
used = pd.Series("", index = df.index)
for i in bad:
    t, rs = try_fix(df.loc[i, "SMILES"])
    if t:
        df.loc[i, "SMILES_fix"] = t
        used[i] = ",".join(rs)

left = [i for i in bad if df.loc[i, "SMILES_fix"] == df.loc[i, "SMILES"]]
print(f"修复成功 {len(bad) - len(left)}/{len(bad)}，未修复：{left}")


# ============ 3. 删除名单 ============
# 歧义修复（SMILES 的化学结构无法唯一确定）
DEL_AMBIG = [701, 1257]
# 同结构但 Td 相差过大（>5°C），判为数据冲突，每条保留一条、删另一条
DEL_TD    = [1256, 1322, 1381]
DEL = set(DEL_AMBIG) | set(DEL_TD)


# ============ 4. 去重（并查集 + Td 容差） ============
TD_TOL = 5.0        # Td 容差（°C）：同一分子的两次录入，Td 差超过它就不合并

def keys_of(mol):
    can  = Chem.MolToSmiles(mol)                          # 含立体规范式
    flat = Chem.MolToSmiles(mol, isomericSmiles = False)  # 去立体规范式
    inch = Chem.MolToInchi(mol) or ("SMI:" + can)
    ann  = (any(a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED for a in mol.GetAtoms())
            or any(b.GetStereo()   != Chem.BondStereo.STEREONONE for b in mol.GetBonds()))
    return {"can": can, "flat": flat, "inchi": inch, "annot": ann}

recs = []
for i in df.index:
    if i in DEL:                                          # 先剔掉已定要删的
        continue
    mol = Chem.MolFromSmiles(df.loc[i, "SMILES_fix"])
    if mol is None:
        print("解析失败，跳过：", i)
        continue
    recs.append({"idx": i, "td": df.loc[i, "Td(°C)"],
                 "smi": df.loc[i, "SMILES_fix"], **keys_of(mol)})

def dedup(recs, level = "safe", tol = TD_TOL):
    n = len(recs)
    p = list(range(n))

    def find(x):
        while p[x] != x:
            p[x] = p[p[x]]                                # 路径压缩
            x = p[x]
        return x

    def td_ok(a, b):
        try:
            return abs(float(recs[a]["td"]) - float(recs[b]["td"])) <= tol
        except (TypeError, ValueError):
            return True                                   # Td 缺失不拦

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb or not td_ok(a, b):
            return
        p[rb] = ra

    first = {}
    for i, r in enumerate(recs):                          # ① InChI 一致
        first.setdefault(r["inchi"], i)
        union(first[r["inchi"]], i)

    review = []
    if level in ("safe", "full"):                         # ②③ 去立体骨架一致
        bk = {}
        for i, r in enumerate(recs):
            bk.setdefault(r["flat"], []).append(i)
        for grp in bk.values():
            if len(grp) < 2:
                continue
            if level == "full":
                for i in grp[1:]:
                    union(grp[0], i)
                continue
            sub = {}
            for i in grp:
                sub.setdefault(recs[i]["inchi"], []).append(i)
            if sum(any(recs[i]["annot"] for i in v) for v in sub.values()) > 1:
                review.append(grp)                        # 双方标了立体且冲突 → 人工
                continue
            for i in grp[1:]:
                union(grp[0], i)

    comp = {}
    for i in range(n):
        comp.setdefault(find(i), []).append(i)

    kept, rm = [], {}
    for grp in comp.values():
        if len(grp) == 1:
            kept.append(recs[grp[0]])
            continue
        # 代表：优先「原始 SMILES 无需修复」的，其次下标最小
        rep = min(grp, key = lambda i: (recs[i]["idx"] in set(bad), recs[i]["idx"]))
        kept.append(recs[rep])
        for i in grp:
            if i != rep:
                rm[recs[i]["idx"]] = recs[rep]["idx"]
    kept.sort(key = lambda r: r["idx"])
    return kept, rm, review

kept, removed, review = dedup(recs)

# 自检：对保留集再跑一次去重，必须删不掉任何东西
_, rem2, _ = dedup(kept)
assert not rem2, f"自检失败，仍有重复：{rem2}"


# ============ 5. 输出 ============
out = pd.DataFrame([{
    "下标":         r["idx"],
    "Td(°C)":       df.loc[r["idx"], "Td(°C)"],
    "SMILES":       r["smi"],
    "标准化SMILES": r["can"],
} for r in kept])

with pd.ExcelWriter(OUT, engine = "openpyxl") as w:
    out.to_excel(w, sheet_name = "去重后", index = False)


# ============ 6. 控制台报告 ============
print("\n" + "=" * 60)
print(f"原始 {len(df)}  →  保留 {len(out)}  →  删除 {len(df) - len(out)}")
print("=" * 60)

print(f"\n[歧义修复，删除 {len(DEL_AMBIG)} 个]")
for i in DEL_AMBIG:
    print(f"  {i}: {df.loc[i, 'SMILES']}")
    print(f"       → {df.loc[i, 'SMILES_fix']}")

print(f"\n[Td 冲突，删除 {len(DEL_TD)} 个]")
for i in DEL_TD:
    peer = {1256: 707, 1322: 436, 1381: 89}[i]            # 与之冲突的另一条
    d = float(df.loc[i, "Td(°C)"]) - float(df.loc[peer, "Td(°C)"])
    print(f"  {i} (Td={df.loc[i,'Td(°C)']})  vs 保留 {peer} (Td={df.loc[peer,'Td(°C)']})  ΔTd={d:+.1f}")

print(f"\n[重复合并，删除 {len(removed)} 个]")
for k, v in sorted(removed.items()):
    print(f"  {k}  →  并入 {v}")

if review:
    print(f"\n[骨架相同但立体冲突，未删，{len(review)} 组]")
    for g in review:
        print("  " + ", ".join(str(recs[i]["idx"]) for i in g))

print(f"\n已保存：{OUT}")

共 1435 条，有问题 19 条
修复成功 19/19，未修复：[]

原始 1435  →  保留 1426  →  删除 9

[歧义修复，删除 2 个]
  701: O=[N+]=Nc1cc(N=[N+]=O)nc(N=[N+]=O)n1
       → ON=Nc1cc(N=NO)nc(N=NO)n1
  1257: Cc1c(N(CN(c2no[n+]([O-])c2C)[N+](=O)O)[N+](=O)[O-])no[n+]1C
       → Cc1c(N(CN(c2no[n+]([O-])c2C)[N+](=O)[O-])[N+](=O)[O-])no[n+]([O-])1

[Td 冲突，删除 3 个]
  1256 (Td=174.0)  vs 保留 707 (Td=273.0)  ΔTd=-99.0
  1322 (Td=257.0)  vs 保留 436 (Td=221.0)  ΔTd=+36.0
  1381 (Td=324.5)  vs 保留 89 (Td=334.1)  ΔTd=-9.6

[重复合并，删除 4 个]
  23  →  并入 1008
  43  →  并入 753
  643  →  并入 420
  1281  →  并入 972

[骨架相同但立体冲突，未删，12 组]
  55, 291, 669, 687
  98, 988, 1207
  126, 727, 1240
  139, 1275
  160, 1325
  226, 418, 855
  241, 520, 1390
  263, 1021, 1264
  378, 649
  477, 918
  1091, 1218
  1095, 1309

已保存：C:\Users\czywg\Desktop\Programmer_czy\chem_project\TDT\out\去重修复结果.xlsx
